# Importar Bibliotecas Requeridas
Importa las bibliotecas necesarias como NumPy, Pandas y scikit-learn para el manejo de datos y el entrenamiento del modelo.

In [1]:
# Importar Bibliotecas Requeridas
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import joblib

# Verificar Valores Faltantes
Verifica si hay valores NaN en el conjunto de datos antes del preprocesamiento.

In [2]:
# Verificar Valores Faltantes
df_check = pd.read_csv('../DARA_PROC/combined_features.csv')
print("Total de valores NaN en el conjunto de datos:", df_check.isna().sum().sum())
print("Valores NaN por columna:")
print(df_check.isna().sum())
print("\nTotal de valores cero en las características:", (df_check[['ECG_mean', 'ECG_std', 'ECG_range', 'ECG_energy', 'ECG_samp_ent', 'ECG_missing_peaks', 'SDNN', 'RMSSD', 'pNN50', 'HR_mean', 'HR_max', 'HR_min']] == 0).sum().sum())

Total de valores NaN en el conjunto de datos: 0
Valores NaN por columna:
UNIX Timestamp       0
DateTime             0
ECG_mean             0
ECG_std              0
ECG_range            0
ECG_energy           0
ECG_samp_ent         0
ECG_missing_peaks    0
SDNN                 0
RMSSD                0
pNN50                0
HR_mean              0
HR_max               0
HR_min               0
FatigueIndex         0
dtype: int64

Total de valores cero en las características: 7404


# Cargar y Preparar Datos
Carga el conjunto de datos desde combined_features.csv y realiza preprocesamiento como manejar valores faltantes y escalado de características.

In [3]:
# Cargar y Preparar Datos
df = pd.read_csv('../DARA_PROC/combined_features.csv')

# Seleccionar características y objetivo
features = ['ECG_mean', 'ECG_std', 'ECG_range', 'ECG_energy', 'ECG_samp_ent', 'ECG_missing_peaks', 'SDNN', 'RMSSD', 'pNN50', 'HR_mean', 'HR_max', 'HR_min']
target = 'FatigueIndex'

X = df[features]
y = df[target]

# Manejar valores faltantes (si los hay)
X = X.fillna(X.mean())
y = y.fillna(y.mean())

# Escalado de características (necesario para SVM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir Datos en Conjuntos de Entrenamiento y Prueba
Usa train_test_split de scikit-learn para dividir los datos en subconjuntos de entrenamiento y prueba.

In [4]:
# Dividir Datos en Conjuntos de Entrenamiento y Prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalado de características (necesario para SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Elegir e Inicializar un Modelo
Selecciona un algoritmo de aprendizaje automático (SVM con kernel RBF) e inicializa el modelo con parámetros predeterminados.

In [5]:
# Elegir e Inicializar un Modelo
model = SVR(kernel='rbf')

# Entrenar el Modelo
Ajusta el modelo a los datos de entrenamiento usando el método fit().

In [6]:
# Entrenar el Modelo
model.fit(X_train_scaled, y_train)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [7]:
# Guardar el Modelo SVM y su escalador
joblib.dump(model, 'svm_regression_model.pkl')
joblib.dump(scaler, 'svm_regression_scaler.pkl')
print("Modelo SVM de regresión guardado como 'svm_regression_model.pkl'")
print("Escalador SVM guardado como 'svm_regression_scaler.pkl'")

Modelo SVM de regresión guardado como 'svm_regression_model.pkl'
Escalador SVM guardado como 'svm_regression_scaler.pkl'


# Evaluar el Rendimiento del Modelo
Evalúa el modelo usando métricas como el error cuadrático medio o R-cuadrado en el conjunto de prueba.

In [8]:
# Evaluar el Rendimiento del Modelo
y_pred = model.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# MSE: Error Cuadrático Medio - menor es mejor (0 es perfecto)
print(f'Mean Squared Error: {mse}')
# RMSE: Raíz del Error Cuadrático Medio - menor es mejor, en unidades del objetivo
print(f'Root Mean Squared Error: {rmse}')
# MAE: Error Absoluto Medio - menor es mejor, menos sensible a outliers que MSE
print(f'Mean Absolute Error: {mae}')
# R²: Coeficiente de Determinación - mayor es mejor (1.0 es perfecto, explica toda la varianza)
print(f'R-squared: {r2}')

Mean Squared Error: 0.6956044761493537
Root Mean Squared Error: 0.8340290619333081
Mean Absolute Error: 0.20816017195035286
R-squared: 0.921208400040324


# Hacer Predicciones
Usa el modelo entrenado para hacer predicciones en datos nuevos o de prueba con el método predict().

In [9]:
# Hacer Predicciones
# Ejemplo: Predecir en datos de prueba
predictions = model.predict(X_test_scaled)
print("Predicciones primeras 5:", predictions[:5])
print("Valores reales primeras 5:", y_test.iloc[:5].values)
print("Errores (Pred - Real) primeras 5:", predictions[:5] - y_test.iloc[:5].values)

Predicciones primeras 5: [-5.35128481 -5.08572748 -5.29128992 -4.74555522 -5.04683251]
Valores reales primeras 5: [-5.40354776 -4.99209565 -5.26389479 -4.723989   -4.97734157]
Errores (Pred - Real) primeras 5: [ 0.05226294 -0.09363182 -0.02739513 -0.02156622 -0.06949094]


# Experimento 2: Entrenamiento sin Características de HR y HRV

In [10]:
# Experimento 2: Entrenar sin características de HR y HRV
# Remover HR_mean, SDNN, RMSSD, pNN50, HR_max, HR_min de X_train y X_test
features_to_remove = ['HR_mean', 'SDNN', 'RMSSD', 'pNN50', 'HR_max', 'HR_min']
X_train_new = X_train.drop(columns=features_to_remove)
X_test_new = X_test.drop(columns=features_to_remove)

# Escalado de características para el experimento 2
scaler2 = StandardScaler()
X_train_new_scaled = scaler2.fit_transform(X_train_new)
X_test_new_scaled = scaler2.transform(X_test_new)

# y_train, y_test permanecen iguales

In [11]:
# Elegir e Inicializar un Modelo (Experimento 2)
model2 = SVR(kernel='rbf')

In [12]:
# Entrenar el Modelo (Experimento 2)
model2.fit(X_train_new_scaled, y_train)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [13]:
# Guardar el Modelo SVM y su escalador (Experimento 2)
joblib.dump(model2, 'svm_regression_model_ablation.pkl')
joblib.dump(scaler2, 'svm_regression_scaler_ablation.pkl')
print("Modelo SVM de ablación guardado como 'svm_regression_model_ablation.pkl'")
print("Escalador SVM de ablación guardado como 'svm_regression_scaler_ablation.pkl'")

Modelo SVM de ablación guardado como 'svm_regression_model_ablation.pkl'
Escalador SVM de ablación guardado como 'svm_regression_scaler_ablation.pkl'


In [14]:
# Evaluar el Rendimiento del Modelo (Experimento 2)
y_pred2 = model2.predict(X_test_new_scaled)
mse2 = mean_squared_error(y_test, y_pred2)
rmse2 = np.sqrt(mse2)
mae2 = mean_absolute_error(y_test, y_pred2)
r22 = r2_score(y_test, y_pred2)

# MSE: Error Cuadrático Medio - menor es mejor (0 es perfecto)
print(f'Mean Squared Error (Exp2): {mse2}')
# RMSE: Raíz del Error Cuadrático Medio - menor es mejor, en unidades del objetivo
print(f'Root Mean Squared Error (Exp2): {rmse2}')
# MAE: Error Absoluto Medio - menor es mejor, menos sensible a outliers que MSE
print(f'Mean Absolute Error (Exp2): {mae2}')
# R²: Coeficiente de Determinación - mayor es mejor (1.0 es perfecto, explica toda la varianza)
print(f'R-squared (Exp2): {r22}')

Mean Squared Error (Exp2): 1.1517202991855793
Root Mean Squared Error (Exp2): 1.0731823233661553
Mean Absolute Error (Exp2): 0.3587555529888747
R-squared (Exp2): 0.8695438454030529


In [15]:
# Hacer Predicciones (Experimento 2)
predictions2 = model2.predict(X_test_new_scaled)
print("Predicciones primeras 5 (Exp2):", predictions2[:5])
print("Valores reales primeras 5:", y_test.iloc[:5].values)
print("Errores (Pred - Real) primeras 5 (Exp2):", predictions2[:5] - y_test.iloc[:5].values)

Predicciones primeras 5 (Exp2): [-5.35400121 -5.04671832 -5.28805042 -4.69277661 -5.00466225]
Valores reales primeras 5: [-5.40354776 -4.99209565 -5.26389479 -4.723989   -4.97734157]
Errores (Pred - Real) primeras 5 (Exp2): [ 0.04954654 -0.05462267 -0.02415563  0.03121239 -0.02732068]
